# RankSEG with the SAM model family

This Colab notebook demonstrates the RankSEG Transformers compatibility layer with SAM1, SAM2, and SAM3 image outputs.

The integration point is always the same: run Hugging Face preprocessing and model inference normally, then replace the final binary mask decision with `rankseg.transformers.postprocess(...)`.

SAM3 is shown last because `facebook/sam3` is gated on Hugging Face.

## 0. Setup

Colab 2026.04 already includes PyTorch 2.10.0, Transformers 5.0.0, Hugging Face Hub 1.8.0, Matplotlib 3.10.0, Pillow 11.3.0, and Requests 2.32.4. This notebook only installs RankSEG from `main`.

In [ ]:
%pip install -q --force-reinstall --no-deps "git+https://github.com/rankseg/rankseg@main"

In [ ]:
import gc
import numpy as np
import requests
import torch
import matplotlib.pyplot as plt
from PIL import Image

from rankseg.transformers import postprocess as rankseg_postprocess


def load_rgb_image(url):
    response = requests.get(url, stream=True, timeout=30)
    response.raise_for_status()
    return Image.open(response.raw).convert("RGB")


def release_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def to_numpy(value):
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().numpy()
    return np.asarray(value)


def first_2d_mask(mask):
    while mask.ndim > 2:
        mask = mask[0]
    return mask


def show_mask(mask, ax, color=(0.12, 0.56, 1.0), alpha=0.55):
    mask = to_numpy(mask).astype(bool)
    overlay = np.zeros((*mask.shape[-2:], 4), dtype=np.float32)
    overlay[..., :3] = np.array(color, dtype=np.float32)
    overlay[..., 3] = mask.astype(np.float32) * alpha
    ax.imshow(overlay)


def show_comparison(image, official_mask, rankseg_mask, title):
    official_mask = to_numpy(official_mask).astype(bool)
    rankseg_mask = to_numpy(rankseg_mask).astype(bool)
    difference = official_mask != rankseg_mask

    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    axes[0].imshow(image)
    axes[0].set_title("Input")
    axes[0].axis("off")

    axes[1].imshow(image)
    show_mask(official_mask, axes[1])
    axes[1].set_title("Official threshold")
    axes[1].axis("off")

    axes[2].imshow(image)
    show_mask(rankseg_mask, axes[2])
    axes[2].set_title("RankSEG")
    axes[2].axis("off")

    axes[3].imshow(difference, cmap="gray", vmin=0, vmax=1)
    axes[3].set_title("Difference")
    axes[3].axis("off")

    fig.suptitle(title)
    plt.tight_layout()


device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)


In [ ]:
image_url = "https://huggingface.co/datasets/hf-internal-testing/fixtures_ade20k/resolve/main/ADE_val_00000001.jpg"
image = load_rgb_image(image_url)
width, height = image.size
input_points = [[[int(width * 0.57), int(height * 0.45)]]]
input_labels = [[1]]

plt.figure(figsize=(7, 7))
plt.imshow(image)
plt.scatter([input_points[0][0][0]], [input_points[0][0][1]], color="lime", marker="*", s=250, edgecolor="white")
plt.title("Input image and point prompt")
plt.axis("off");

## 1. SAM1 prompt masks

SAM1 removes processor padding before resizing masks back to the original image size. The Hugging Face model is run once, then the official post-processing and RankSEG post-processing reuse the same outputs.

In [ ]:
from transformers import SamModel, SamProcessor

sam1_checkpoint = "facebook/sam-vit-base"
sam1_processor = SamProcessor.from_pretrained(sam1_checkpoint)
sam1_model = SamModel.from_pretrained(sam1_checkpoint).to(device).eval()

sam1_inputs = sam1_processor(
    image,
    input_points=input_points,
    input_labels=input_labels,
    return_tensors="pt",
).to(device)
sam1_original_sizes = sam1_inputs["original_sizes"].cpu()
sam1_reshaped_input_sizes = sam1_inputs["reshaped_input_sizes"].cpu()

with torch.inference_mode():
    sam1_outputs = sam1_model(**sam1_inputs)

# The model is no longer needed after outputs are materialized.
del sam1_model, sam1_inputs
release_cuda()

print("SAM1 pred_masks:", tuple(sam1_outputs.pred_masks.shape))


In [ ]:
# Official Hugging Face SAM1 mask post-processing from the same sam1_outputs.
sam1_official = sam1_processor.image_processor.post_process_masks(
    sam1_outputs.pred_masks.cpu(),
    sam1_original_sizes,
    sam1_reshaped_input_sizes,
)[0][0]


In [ ]:
# The original inference flow above stays unchanged. RankSEG only replaces the final post-processing call:
# sam1_inputs = sam1_processor(image, input_points=input_points, input_labels=input_labels, return_tensors="pt").to(device)
# with torch.inference_mode():
#     sam1_outputs = sam1_model(**sam1_inputs)
sam1_rankseg = rankseg_postprocess(
    sam1_outputs,
    original_sizes=sam1_original_sizes,
    reshaped_input_sizes=sam1_reshaped_input_sizes,
    rankseg_kwargs={"metric": "dice", "solver": "RMA"},
)[0][0]

show_comparison(image, first_2d_mask(sam1_official), first_2d_mask(sam1_rankseg), "SAM1")


## 2. SAM2 prompt masks

SAM2 prompt mask post-processing resizes masks directly to `original_sizes`. The Hugging Face model is run once, then the official post-processing and RankSEG post-processing reuse the same outputs.

In [ ]:
from transformers import AutoModelForMaskGeneration, Sam2Processor

sam2_checkpoint = "facebook/sam2-hiera-base-plus"
sam2_processor = Sam2Processor.from_pretrained(sam2_checkpoint)
sam2_model = AutoModelForMaskGeneration.from_pretrained(sam2_checkpoint).to(device).eval()
sam2_input_points = [[input_points[0]]]
sam2_input_labels = [[input_labels[0]]]

sam2_inputs = sam2_processor(
    images=image,
    input_points=sam2_input_points,
    input_labels=sam2_input_labels,
    return_tensors="pt",
).to(device)
sam2_original_sizes = sam2_inputs["original_sizes"].cpu()

with torch.inference_mode():
    sam2_outputs = sam2_model(**sam2_inputs)

# The model is no longer needed after outputs are materialized.
del sam2_model, sam2_inputs
release_cuda()

print("SAM2 pred_masks:", tuple(sam2_outputs.pred_masks.shape))


In [ ]:
# Official Hugging Face SAM2 mask post-processing from the same sam2_outputs.
sam2_official = sam2_processor.post_process_masks(
    sam2_outputs.pred_masks.cpu(),
    sam2_original_sizes,
)[0][0]


In [ ]:
# The original inference flow above stays unchanged. RankSEG only replaces the final post-processing call:
# sam2_inputs = sam2_processor(images=image, input_points=sam2_input_points, input_labels=sam2_input_labels, return_tensors="pt").to(device)
# with torch.inference_mode():
#     sam2_outputs = sam2_model(**sam2_inputs)
sam2_rankseg = rankseg_postprocess(
    sam2_outputs,
    original_sizes=sam2_original_sizes,
    rankseg_kwargs={"metric": "dice", "solver": "RMA"},
)[0][0]

show_comparison(image, first_2d_mask(sam2_official), first_2d_mask(sam2_rankseg), "SAM2")


## 3. SAM3 gated model login

`facebook/sam3` is gated. Run the next cell with a Hugging Face account that has access to the model.

In [ ]:
from huggingface_hub import login, notebook_login

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None

if hf_token:
    login(token=hf_token)
else:
    notebook_login()

## 4. SAM3 instance and semantic masks

SAM3 is split into one Hugging Face inference cell followed by official and RankSEG post-processing cells. Change `text_prompt` and rerun only the SAM3 inference/post-processing cells; the model is deleted right after inference to keep Colab GPU memory available.

In [ ]:
from transformers import Sam3Model, Sam3Processor

sam3_checkpoint = "facebook/sam3"
sam3_processor = Sam3Processor.from_pretrained(sam3_checkpoint, token=True)
sam3_model = Sam3Model.from_pretrained(sam3_checkpoint, device_map="auto", token=True).eval()

text_prompt = "house"
sam3_inputs = sam3_processor(images=image, text=text_prompt, return_tensors="pt").to(sam3_model.device)
sam3_target_sizes = sam3_inputs["original_sizes"].cpu().tolist()
sam3_original_size = tuple(int(v) for v in sam3_target_sizes[0])

with torch.inference_mode():
    sam3_outputs = sam3_model(**sam3_inputs)

# The large model is no longer needed after outputs are materialized.
del sam3_model, sam3_inputs
release_cuda()

sam3_score_threshold = 0.3
sam3_mask_threshold = 0.5
print("text_prompt:", text_prompt)
print("original_size:", sam3_original_size)
print("pred_masks:", tuple(sam3_outputs.pred_masks.shape))
print("semantic_seg:", tuple(sam3_outputs.semantic_seg.shape))


In [ ]:
# Official Hugging Face instance post-processing from the same sam3_outputs.
sam3_official_instances = sam3_processor.post_process_instance_segmentation(
    sam3_outputs,
    threshold=sam3_score_threshold,
    mask_threshold=sam3_mask_threshold,
    target_sizes=sam3_target_sizes,
)[0]


In [ ]:
# The original inference flow above stays unchanged. RankSEG only replaces the final post-processing call:
# sam3_model = Sam3Model.from_pretrained(sam3_checkpoint, device_map="auto", token=True).eval()
# sam3_inputs = sam3_processor(images=image, text=text_prompt, return_tensors="pt").to(sam3_model.device)
# with torch.inference_mode():
#     sam3_outputs = sam3_model(**sam3_inputs)
sam3_rankseg_instances = rankseg_postprocess(
    sam3_outputs,
    threshold=sam3_score_threshold,
    target_sizes=sam3_target_sizes,
    rankseg_kwargs={"metric": "dice", "solver": "RMA"},
)[0]

empty_sam3 = torch.zeros(sam3_original_size, dtype=torch.long)
official_instance_mask = (
    sam3_official_instances["masks"][0].detach().cpu()
    if len(sam3_official_instances["masks"])
    else empty_sam3
)
rankseg_instance_mask = (
    sam3_rankseg_instances["masks"][0].detach().cpu()
    if len(sam3_rankseg_instances["masks"])
    else empty_sam3
)

show_comparison(image, official_instance_mask, rankseg_instance_mask, "SAM3 instance")


In [ ]:
# Official Hugging Face semantic post-processing from the same sam3_outputs.
sam3_official_semantic = sam3_processor.post_process_semantic_segmentation(
    sam3_outputs,
    target_sizes=sam3_target_sizes,
    threshold=sam3_mask_threshold,
)[0].detach().cpu().to(torch.long)

# The original inference flow above stays unchanged. RankSEG only replaces the final post-processing call:
# sam3_model = Sam3Model.from_pretrained(sam3_checkpoint, device_map="auto", token=True).eval()
# sam3_inputs = sam3_processor(images=image, text=text_prompt, return_tensors="pt").to(sam3_model.device)
# with torch.inference_mode():
#     sam3_outputs = sam3_model(**sam3_inputs)
sam3_rankseg_semantic = rankseg_postprocess(
    sam3_outputs,
    sam_task="semantic",
    target_sizes=sam3_original_size,
    rankseg_kwargs={"metric": "dice", "solver": "RMA"},
)[0].detach().cpu().to(torch.long)

show_comparison(image, sam3_official_semantic, sam3_rankseg_semantic, "SAM3 semantic")
